# 00 · Create Unity Catalog schemas and landing Volumes

Bootstraps every Unity Catalog object the Blue Ridge LNG PM Co-Pilot build writes to, so later notebooks and pipelines never create schemas themselves.

| Object | Count | Notes |
|---|---|---|
| Schemas | 7 | all prefixed `bluerridge_`, inside the existing catalog `mfg_mc_se_sa` (`CREATE CATALOG` is blocked in this workspace) |
| Volumes | 2 | `drawings` and `project`, both managed Volumes in `bluerridge_landing` |

Every statement is `IF NOT EXISTS`, so re-running changes nothing. The last cell writes `outputs/schemas_listing.md` next to this notebook as the committed evidence.

In [1]:
# Widgets let a job override these; the defaults match the variables in databricks.yml.
dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")

catalog = dbutils.widgets.get("catalog")
prefix = dbutils.widgets.get("schema_prefix")

# Schema suffix -> purpose (spec section 6.2.2). No apostrophes: these go into SQL COMMENT '...'.
SCHEMAS = {
    "landing": "Raw landing zone: Volumes holding the drawing PDFs and the project CSVs",
    "bronze_drawings": "Bronze: raw drawing PDF binaries plus Auto Loader metadata",
    "bronze_project": "Bronze: raw project tables (schedule, cost, NCRs, change orders, safety log, daily reports)",
    "silver_engineering": "Silver: cleaned project tables plus parsed and extracted drawings",
    "gold_pm": "Gold: PM-facing aggregates (compliance findings, KPIs, risk forecast, daily brief)",
    "agents": "Genie Agent and Vector Search artifacts (code excerpts, prompt configs)",
    "app": "Databricks App working tables, if any",
}

# Volume name -> purpose. Both live in <prefix>_landing.
VOLUMES = {
    "drawings": "Synthetic engineering drawing PDFs (ISO, P&ID, GA, spec sheets)",
    "project": "Synthetic project data CSVs (schedule, cost, NCRs, change orders, safety log, daily reports)",
}

print(f"catalog={catalog}  prefix={prefix}  ->  {len(SCHEMAS)} schemas, {len(VOLUMES)} volumes")

catalog=mfg_mc_se_sa  prefix=bluerridge  ->  7 schemas, 2 volumes


## 1. Schemas

In [2]:
for suffix, purpose in SCHEMAS.items():
    schema = f"{prefix}_{suffix}"
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}` COMMENT '{purpose}'")
    print(f"schema  {catalog}.{schema}")

schema  mfg_mc_se_sa.bluerridge_landing
schema  mfg_mc_se_sa.bluerridge_bronze_drawings
schema  mfg_mc_se_sa.bluerridge_bronze_project
schema  mfg_mc_se_sa.bluerridge_silver_engineering
schema  mfg_mc_se_sa.bluerridge_gold_pm
schema  mfg_mc_se_sa.bluerridge_agents
schema  mfg_mc_se_sa.bluerridge_app


## 2. Volumes

In [3]:
landing = f"{prefix}_landing"
for volume, purpose in VOLUMES.items():
    spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{landing}`.`{volume}` COMMENT '{purpose}'")
    print(f"volume  /Volumes/{catalog}/{landing}/{volume}")

volume  /Volumes/mfg_mc_se_sa/bluerridge_landing/drawings
volume  /Volumes/mfg_mc_se_sa/bluerridge_landing/project


## 3. Verify and write the evidence listing

`SHOW SCHEMAS ... LIKE` treats `*` (not `%`) as its wildcard and matches `%` literally, so the filter is `'bluerridge_*'`. The assertions fail the run if any schema or Volume is missing.

In [4]:
found = [r[0] for r in spark.sql(f"SHOW SCHEMAS IN `{catalog}` LIKE '{prefix}_*'").collect()]
expected = {f"{prefix}_{suffix}" for suffix in SCHEMAS}
assert set(found) == expected, f"schema mismatch: missing={expected - set(found)} unexpected={set(found) - expected}"

schemas_df = spark.sql(f"""
    SELECT schema_name, schema_owner, comment, created
    FROM `{catalog}`.information_schema.schemata
    WHERE startswith(schema_name, '{prefix}_')
    ORDER BY schema_name
""")
volumes_df = spark.sql(f"""
    SELECT volume_schema, volume_name, volume_type, volume_owner, comment
    FROM `{catalog}`.information_schema.volumes
    WHERE volume_schema = '{landing}'
    ORDER BY volume_name
""")
assert {r.volume_name for r in volumes_df.collect()} == set(VOLUMES), "volume mismatch"

display(schemas_df)
display(volumes_df)

schema_name,schema_owner,comment,created
bluerridge_agents,joseph.hobbs@databricks.com,"Genie Agent and Vector Search artifacts (code excerpts, prompt configs)",2026-11-06T23:32:32.060Z
bluerridge_app,joseph.hobbs@databricks.com,"Databricks App working tables, if any",2026-11-06T23:32:32.499Z
bluerridge_bronze_drawings,joseph.hobbs@databricks.com,Bronze: raw drawing PDF binaries plus Auto Loader metadata,2026-11-06T23:32:30.118Z
bluerridge_bronze_project,joseph.hobbs@databricks.com,"Bronze: raw project tables (schedule, cost, NCRs, change orders, safety log, daily reports)",2026-11-06T23:32:30.852Z
bluerridge_gold_pm,joseph.hobbs@databricks.com,"Gold: PM-facing aggregates (compliance findings, KPIs, risk forecast, daily brief)",2026-11-06T23:32:31.646Z
bluerridge_landing,joseph.hobbs@databricks.com,Raw landing zone: Volumes holding the drawing PDFs and the project CSVs,2026-11-06T23:32:29.352Z
bluerridge_silver_engineering,joseph.hobbs@databricks.com,Silver: cleaned project tables plus parsed and extracted drawings,2026-11-06T23:32:31.271Z


volume_schema,volume_name,volume_type,volume_owner,comment
bluerridge_landing,drawings,MANAGED,joseph.hobbs@databricks.com,"Synthetic engineering drawing PDFs (ISO, P&ID, GA, spec sheets)"
bluerridge_landing,project,MANAGED,joseph.hobbs@databricks.com,"Synthetic project data CSVs (schedule, cost, NCRs, change orders, safety log, daily reports)"


In [5]:
from datetime import datetime, timezone
from pathlib import Path

stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
user = spark.sql("SELECT current_user()").first()[0]

lines = [
    "# Unity Catalog bootstrap: schemas and Volumes",
    "",
    f"Written by `00_setup/00_create_schemas_and_volumes.ipynb` on {stamp} as {user}.",
    "",
    f"## Schemas: `SHOW SCHEMAS IN {catalog} LIKE '{prefix}_*'` returned {len(found)} rows",
    "",
    "| Schema | Owner | Comment |",
    "|---|---|---|",
]
for r in schemas_df.collect():
    lines.append(f"| `{catalog}.{r.schema_name}` | {r.schema_owner} | {r.comment} |")
lines += [
    "",
    f"## Volumes in `{catalog}.{landing}`",
    "",
    "| Volume | Type | Path | Comment |",
    "|---|---|---|---|",
]
for r in volumes_df.collect():
    lines.append(
        f"| `{r.volume_name}` | {r.volume_type} | `/Volumes/{catalog}/{r.volume_schema}/{r.volume_name}` | {r.comment} |"
    )

listing = "\n".join(lines) + "\n"
out = Path("outputs") / "schemas_listing.md"
out.parent.mkdir(exist_ok=True)
out.write_text(listing)
print(listing)

# Unity Catalog bootstrap: schemas and Volumes

Written by `00_setup/00_create_schemas_and_volumes.ipynb` on 2026-11-06 23:32 UTC as joseph.hobbs@databricks.com.

## Schemas: `SHOW SCHEMAS IN mfg_mc_se_sa LIKE 'bluerridge_*'` returned 7 rows

| Schema | Owner | Comment |
|---|---|---|
| `mfg_mc_se_sa.bluerridge_agents` | joseph.hobbs@databricks.com | Genie Agent and Vector Search artifacts (code excerpts, prompt configs) |
| `mfg_mc_se_sa.bluerridge_app` | joseph.hobbs@databricks.com | Databricks App working tables, if any |
| `mfg_mc_se_sa.bluerridge_bronze_drawings` | joseph.hobbs@databricks.com | Bronze: raw drawing PDF binaries plus Auto Loader metadata |
| `mfg_mc_se_sa.bluerridge_bronze_project` | joseph.hobbs@databricks.com | Bronze: raw project tables (schedule, cost, NCRs, change orders, safety log, daily reports) |
| `mfg_mc_se_sa.bluerridge_gold_pm` | joseph.hobbs@databricks.com | Gold: PM-facing aggregates (compliance findings, KPIs, risk forecast, daily brief) |
| `mfg_mc_